In [ ]:
# Clean old conflicting libraries
!pip uninstall -y transformers adapter-transformers adapters tokenizers huggingface_hub

# Install stable stack
!pip install transformers==4.40.2
!pip install adapters
!pip install datasets
!pip install seqeval

Found existing installation: transformers 4.40.2
Uninstalling transformers-4.40.2:
  Successfully uninstalled transformers-4.40.2
Found existing installation: tokenizers 0.19.1
Uninstalling tokenizers-0.19.1:
  Successfully uninstalled tokenizers-0.19.1
Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2
  Using cached transformers-4.40.2-py3-none-any.whl.metadata (137 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.40.2-py3-none-any.whl (9.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
Using cached tokenizers-0.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source

  Using cached adapters-1.2.0-py3-none-any.whl.metadata (17 kB)
  Using cached transformers-4.51.3-py3-none-any.whl.metadata (38 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached adapters-1.2.0-py3-none-any.whl (302 kB)
Using cached transformers-4.51.3-py3-none-any.whl (10.4 MB)
Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.40.2
    Uninstalling transformers-4.40.2:
      Successfully uninstalled transformers-4.40.2


In [ ]:
import torch
import numpy as np

from datasets import load_dataset
from torch.utils.data import DataLoader

from transformers import BertTokenizerFast
from adapters import BertAdapterModel
from adapters.composition import Fuse

from seqeval.metrics import classification_report

In [ ]:
model_name = "bert-base-multilingual-cased"

tokenizer = BertTokenizerFast.from_pretrained(model_name)

model = BertAdapterModel.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertAdapterModel were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['heads.default.3.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def convert_ud(input_file,output_file):

    with open(input_file) as f, open(output_file,"w") as out:

        for line in f:

            if line.startswith("#") or line.strip()=="":
                out.write("\n")
                continue

            parts=line.split("\t")

            if "-" in parts[0] or "." in parts[0]:
                continue

            word=parts[1]
            pos=parts[3]

            out.write(word+" "+pos+"\n")


for lang in datasets.keys():

    convert_ud(
        f"data/{lang}.conllu",
        f"data/{lang}.conll"
    )

print("Conversion to .conll format done")

Conversion to .conll format done


In [ ]:
label_list = [
"ADJ","ADP","ADV","AUX","CCONJ","DET","INTJ",
"NOUN","NUM","PART","PRON","PROPN","PUNCT",
"SCONJ","SYM","VERB","X"
]

label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}

In [ ]:
model.add_tagging_head(
    "pos_head",
    num_labels=len(label_list),
    id2label=id2label
)

model.active_head = "pos_head"

In [ ]:
from datasets import Dataset

def load_conll_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = f.read()

    sentences_raw = data.strip().split("\n\n")
    all_tokens = []
    all_tags = []

    for sentence_str in sentences_raw:
        tokens = []
        tags = []
        for line in sentence_str.split("\n"):
            if line.strip():
                word, tag = line.split(" ")
                tokens.append(word)
                tags.append(tag)
        if tokens:
            all_tokens.append(tokens)
            all_tags.append(tags)

    return Dataset.from_dict({"tokens": all_tokens, "tags": all_tags})

hindi = load_conll_data("data/hindi.conll")
tamil = load_conll_data("data/tamil.conll")
telugu = load_conll_data("data/telugu.conll")
marathi = load_conll_data("data/marathi.conll")

In [ ]:
import os
import requests

datasets = {
"hindi":"https://raw.githubusercontent.com/UniversalDependencies/UD_Hindi-HDTB/master/hi_hdtb-ud-train.conllu",
"telugu":"https://raw.githubusercontent.com/UniversalDependencies/UD_Telugu-MTG/master/te_mtg-ud-train.conllu",
"tamil":"https://raw.githubusercontent.com/UniversalDependencies/UD_Tamil-TTB/master/ta_ttb-ud-train.conllu",
"marathi":"https://raw.githubusercontent.com/UniversalDependencies/UD_Marathi-UFAL/master/mr_ufal-ud-train.conllu"
}

os.makedirs("data",exist_ok=True)

for lang,url in datasets.items():

    r=requests.get(url)

    with open(f"data/{lang}.conllu","wb") as f:
        f.write(r.content)

print("Universal Dependencies datasets downloaded")

Universal Dependencies datasets downloaded


In [ ]:
def tokenize(example):

    tokens = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True
    )

    word_ids = tokens.word_ids()

    labels = []
    previous = None

    for word_id in word_ids:

        if word_id is None:
            labels.append(-100)

        elif word_id != previous:
            # Convert string tag to its numerical ID
            labels.append(label2id[example["tags"][word_id]])

        else:
            labels.append(-100)

        previous = word_id

    tokens["labels"] = labels

    return tokens

In [ ]:
hindi = hindi.map(tokenize, remove_columns=["tokens", "tags"])
tamil = tamil.map(tokenize, remove_columns=["tokens", "tags"])
telugu = telugu.map(tokenize, remove_columns=["tokens", "tags"])
marathi = marathi.map(tokenize, remove_columns=["tokens", "tags"])

Map:   0%|          | 0/13306 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/1051 [00:00<?, ? examples/s]

Map:   0%|          | 0/373 [00:00<?, ? examples/s]

In [ ]:
languages = ["hindi","telugu","tamil","marathi"]

for lang in languages:
    model.add_adapter(lang, overwrite_ok=True)

model.to(device)
print("Using device:", device)

Using device: cuda


In [ ]:
from transformers import DataCollatorForTokenClassification

def train_adapter(lang, dataset):

    print("Training adapter:", lang)

    model.train_adapter(lang)

    data_collator = DataCollatorForTokenClassification(tokenizer, label_pad_token_id=-100)

    loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=data_collator)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=5e-5
    )

    model.train()

    for epoch in range(1):

        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

    print(lang,"done")

In [ ]:
train_adapter("hindi",hindi)
train_adapter("telugu",telugu)
train_adapter("tamil",tamil)
train_adapter("marathi",marathi)

Training adapter: hindi
hindi done
Training adapter: telugu
telugu done
Training adapter: tamil
tamil done
Training adapter: marathi
marathi done


In [ ]:
adapter_names = ["hindi","telugu","tamil","marathi"]

model.add_adapter_fusion(adapter_names, overwrite_ok=True)

model.set_active_adapters(
    Fuse(*adapter_names)
)

model.train_adapter_fusion(adapter_names)

/usr/local/lib/python3.12/dist-packages/adapters/composition.py:243: FutureWarning: Passing list objects for adapter activation is deprecated. Please use Stack or Fuse explicitly.
  warnings.warn(


In [ ]:
import pandas as pd

gondi = pd.read_csv("gondi.conll", sep=" ", names=["word","pos"])

print(gondi.head())

    word   pos
0   penu  NOUN
1   penu  NOUN
2    pad  VERB
3   nang  PRON
4  manus  NOUN
